# OneSite Full Pipeline
Descarga ZIPs desde SFTP, extrae Excel, divide por hojas y produce 6 tablas de salida.

**SFTP:** `emft.realpage.com`
**Salidas en** `Onesite_Examples/tables/`:
`ar_collections`, `denies_cancells`, `effective_rents`, `move_ins`, `deliquent_prepaid`, `leasing_desk`

**Prerrequisito:** `pip install paramiko openpyxl pandas`

In [ ]:
# %pip install paramiko openpyxl pandas

In [ ]:
import os, re, stat, zipfile, pathlib, datetime as dt, warnings, math
import pandas as pd
import paramiko
warnings.filterwarnings('ignore')

# SFTP
SFTP_HOST      = "emft.realpage.com"
SFTP_PORT      = 22
SFTP_USER      = "EXT-3712076_OS"
SFTP_PASS      = "7QV08u8KHLWtnU"
SFTP_PRIMARY   = "/os_home/ReportGroups/Fabric Reports"
SFTP_SECONDARY = "/os_home/ReportGroups/Fabric Reports 2"

# Local paths
BASE_DIR  = pathlib.Path(os.path.dirname(os.path.abspath("__file__")))
ZIP_DIR   = BASE_DIR / "_zips"
RAW_DIR   = BASE_DIR / "xlsx_by_sheet" / "_raw"
SHEET_DIR = BASE_DIR / "xlsx_by_sheet"
TABLE_DIR = BASE_DIR / "tables"
for d in [ZIP_DIR, RAW_DIR, SHEET_DIR, TABLE_DIR]:
    d.mkdir(parents=True, exist_ok=True)

# Date helpers
now       = dt.datetime.now()
M, D, Y   = now.month, now.day, now.year
YYYYMMDD  = f"{Y}{M:02d}{D:02d}"
TODAY     = dt.date.today()
YESTERDAY = TODAY - dt.timedelta(days=1)

primary_rx   = re.compile(rf"^SpecificDate_SystemDate_{M}_{D}_{Y}.*\.zip$", re.I)
secondary_rx = re.compile(
    rf"^(?:OnDemand_.*{YYYYMMDD}.*\.zip|SpecificDate_SystemDate_{M}_{D}_{Y}.*\.zip)$", re.I
)

EXCLUDE_PROPS = {"BBI LLC - Single Familiy", "First Property I LLC - Single Family", "Zoe LLC"}

print(f"BASE_DIR  : {BASE_DIR}")
print(f"Fecha     : {M}/{D}/{Y}  ({YYYYMMDD})")
print(f"YESTERDAY : {YESTERDAY}")

In [ ]:
# ─── General helpers ──────────────────────────────────────────────────────

def _long_path(path):
    s = str(pathlib.Path(path).resolve())
    if os.name == "nt":
        bs = chr(92)
        prefix = bs + bs + "?" + bs
        if not s.startswith(prefix):
            s = prefix + s
    return s

def list_matching_zips(sftp, folder, rx):
    matches = []
    try:
        for entry in sftp.listdir_attr(folder):
            if stat.S_ISREG(entry.st_mode) and rx.match(entry.filename):
                matches.append((folder, entry.filename))
    except FileNotFoundError:
        print(f"  Carpeta no encontrada: {folder}")
    return matches

def extract_zip(zip_path, out_dir):
    out_dir = pathlib.Path(out_dir)
    extracted, skipped = [], []
    with zipfile.ZipFile(zip_path, "r") as z:
        for member in z.infolist():
            if member.filename.endswith("/"):
                continue
            target = out_dir / member.filename
            try:
                target.parent.mkdir(parents=True, exist_ok=True)
                with z.open(member) as src, open(_long_path(target), "wb") as dst:
                    dst.write(src.read())
                extracted.append(target)
            except Exception as e:
                print(f"    SKIP {member.filename}: {e}")
                skipped.append(member.filename)
    return extracted

def slug(s):
    s = re.sub(r"[^a-z0-9]+", "_", str(s).lower().strip())
    return s.strip("_")

def normalize_base(filename):
    base = os.path.splitext(filename)[0].strip()
    base = re.sub(r"^\d{7}_", "", base)
    base = re.sub(r"_\d{7}$", "", base)
    return base

def to_num(x):
    try:
        v = str(x).replace(",", "").replace("$", "").replace("(", "-").replace(")", "").strip()
        return float(v)
    except Exception:
        return 0.0

def safe_float(x):
    try:
        if pd.isna(x): return None
        v = str(x).replace(",", "").replace("$", "").replace("(", "-").replace(")", "").strip()
        f = float(v)
        return None if math.isnan(f) or math.isinf(f) else f
    except Exception:
        return None

def safe_to_date(x):
    try:
        if pd.isna(x): return None
        return pd.to_datetime(x).date()
    except Exception:
        return None

def find_col(df, *keywords):
    kws = [k.lower() for k in keywords]
    for i, c in enumerate(df.columns):
        cs = str(c).lower()
        if any(k in cs for k in kws):
            return df.columns[i]
    return None

def find_header_row(df_raw, token, max_rows=200):
    for i in range(min(max_rows, len(df_raw))):
        row = df_raw.iloc[i].astype(str).tolist()
        if any(token.lower() in v.lower() for v in row):
            return i
    return None

def join_key(x):
    if x is None or pd.isna(x): return ""
    return str(x).strip().upper()

print("Helpers listos.")

In [ ]:
downloaded_zips = []   # (local_path, original_name, idx)
failed_zips = []

with paramiko.Transport((SFTP_HOST, SFTP_PORT)) as tr:
    tr.connect(username=SFTP_USER, password=SFTP_PASS)
    print("Conectado:", SFTP_HOST)

    with paramiko.SFTPClient.from_transport(tr) as sftp:
        primary_matches   = list_matching_zips(sftp, SFTP_PRIMARY, primary_rx)
        secondary_matches = list_matching_zips(sftp, SFTP_SECONDARY, secondary_rx)

        if not primary_matches:
            print("ADVERTENCIA: No hay ZIPs de hoy en Primary. Disponibles:")
            try:
                for e in sftp.listdir(SFTP_PRIMARY):
                    if e.lower().endswith(".zip"): print(f"  {e}")
            except Exception as ex:
                print(f"  Error: {ex}")

        # Deduplicar por nombre (primary gana)
        all_matches = {}
        for folder, fname in (primary_matches + secondary_matches):
            all_matches.setdefault(fname, folder)

        print(f"Total ZIPs: {len(all_matches)}")

        for idx, (fname, folder) in enumerate(all_matches.items(), start=1):
            local_zip = ZIP_DIR / f"zip_{idx}.zip"
            try:
                print(f"  [{idx}] {fname}")
                sftp.get(f"{folder}/{fname}", str(local_zip))
                downloaded_zips.append((local_zip, fname, idx))
                print(f"       OK -> zip_{idx}.zip")
            except Exception as e:
                print(f"       ERROR: {e}")
                failed_zips.append((fname, str(e)))

print(f"\nDescargados: {len(downloaded_zips)}  Fallidos: {len(failed_zips)}")

In [ ]:
all_excel_files = []

# ─── Extract ZIPs ──────────────────────────────────────────────────────────
for local_zip, orig_name, idx in downloaded_zips:
    sub_dir = RAW_DIR / f"zip_{idx}"
    sub_dir.mkdir(parents=True, exist_ok=True)
    print(f"Extrayendo [{idx}]: {orig_name}")
    try:
        extracted = extract_zip(local_zip, sub_dir)
        xl_files = [p for p in extracted if p.suffix.lower() in (".xlsx", ".xls")]
        all_excel_files.extend(xl_files)
        print(f"  {len(xl_files)} Excel(s)")
        try: local_zip.unlink(missing_ok=True)
        except: pass
    except Exception as e:
        print(f"  ERROR extrayendo: {e}")

# ─── Split each Excel by sheet ─────────────────────────────────────────────
split_files = []
for xl_path in all_excel_files:
    base_slug_name = slug(normalize_base(xl_path.name))
    try:
        xl = pd.ExcelFile(_long_path(xl_path))
        for sheet in xl.sheet_names:
            sheet_slug_name = slug(sheet)
            out_name = f"{base_slug_name}__{sheet_slug_name}.xlsx"
            out_path = SHEET_DIR / out_name
            try:
                df = xl.parse(sheet, header=None)
                df.to_excel(_long_path(out_path), index=False, header=False, engine="openpyxl")
                split_files.append(out_path)
            except Exception as e:
                print(f"  SKIP {out_name}: {e}")
    except Exception as e:
        print(f"  ERROR abriendo {xl_path.name}: {e}")

print(f"\nTotal hojas generadas: {len(split_files)}")
for p in sorted(split_files, key=lambda x: x.name):
    print(f"  {p.name}")

In [ ]:
# ═══════════════════════════════════════════════════════
#  ar_collections
#  Delinquent Sheet4 + Resident Balances Sheet1
# ═══════════════════════════════════════════════════════
HEADER_ROW_IDX = 2

EXCLUDE_NAMES = {
    "bbi_llc", "first_property_i_llc", "zoe_llc"
}

def _excl(fname):
    fl = fname.lower()
    return any(fl.startswith(e) for e in EXCLUDE_NAMES)

delinq_files = [
    p for p in SHEET_DIR.iterdir()
    if p.is_file()
    and "delinquent_and_prepaid" in p.name.lower()
    and "sheet4" in p.name.lower()
    and not _excl(p.name)
]

billing_files = [
    p for p in SHEET_DIR.iterdir()
    if p.is_file()
    and "resident_balances_by_fiscal_period" in p.name.lower()
    and "sheet1" in p.name.lower()
    and not _excl(p.name)
]

print(f"Delinquent Sheet4: {len(delinq_files)}")
print(f"Billing Sheet1   : {len(billing_files)}")

def prop_from_delinq(fname):
    return re.sub(r"_delinquent_and_prepaid.*", "", fname, flags=re.I).replace("_", " ").strip()

def prop_from_bill(fname):
    return re.sub(r"_resident_balances.*", "", fname, flags=re.I).replace("_", " ").strip()

delinq_rows = []
for fp in delinq_files:
    try:
        raw = pd.read_excel(_long_path(fp), header=None, engine="openpyxl")
        prop = prop_from_delinq(fp.stem)

        df = raw.iloc[HEADER_ROW_IDX:].copy()
        df.columns = df.iloc[0].astype(str)
        df = df.iloc[1:].reset_index(drop=True)
        df.columns = [c.strip() for c in df.columns]

        # Filter out totals and junk rows
        col2 = df.iloc[:, 1].astype(str)
        df = df[col2.notna() & (col2 != "2130.0") & (col2 != "Delinquent/Prepaid Account")]
        df = df[~df.iloc[:, 0].astype(str).str.startswith("Total")]

        c_curr = find_col(df, "Current")
        c_30   = find_col(df, "30 Days", "30Days")
        c_60   = find_col(df, "60 Days", "60Days")
        c_90   = find_col(df, "90+ Days", "90+Days", "90 Days")

        if not all([c_curr, c_30, c_60, c_90]):
            print(f"  WARN {fp.name}: faltan columnas financieras")
            continue

        current   = df[c_curr].apply(to_num).sum()
        days30    = df[c_30].apply(to_num).sum()
        days60    = df[c_60].apply(to_num).sum()
        days90    = df[c_90].apply(to_num).sum()

        delinq_rows.append({
            "PropertyName": prop,
            "Current": current,
            "Days30Plus": days30 + days60 + days90,
            "RunDate": str(YESTERDAY),
        })
        print(f"  OK  {fp.name} -> {prop}")
    except Exception as e:
        print(f"  ERR {fp.name}: {e}")

billing_rows = []
for fp in billing_files:
    try:
        raw = pd.read_excel(_long_path(fp), header=None, engine="openpyxl")
        prop = prop_from_bill(fp.stem)

        mask = raw.iloc[:, 1].astype(str).str.strip().str.lower() == "totals"
        totals = raw[mask]
        if totals.empty or raw.shape[1] <= 24:
            billing_val = None
        else:
            billing_val = safe_float(totals.iloc[0, 24])

        billing_rows.append({"PropertyName": prop, "CurrentMonthBillings": billing_val})
        print(f"  OK  {fp.name} -> {prop}  billing={billing_val}")
    except Exception as e:
        print(f"  ERR {fp.name}: {e}")

df_delinq = pd.DataFrame(delinq_rows)
df_bill   = pd.DataFrame(billing_rows)

if not df_delinq.empty:
    df_delinq["_key"] = df_delinq["PropertyName"].str.upper().str.strip()
    if not df_bill.empty:
        df_bill["_key"] = df_bill["PropertyName"].str.upper().str.strip()
        merged = df_delinq.merge(df_bill[["_key","CurrentMonthBillings"]], on="_key", how="left")
    else:
        merged = df_delinq.copy()
        merged["CurrentMonthBillings"] = None
    merged["CurrentMonthBillingsNotCollected"] = merged.apply(
        lambda r: round(r["Current"] / r["CurrentMonthBillings"], 4)
        if (r["CurrentMonthBillings"] and r["CurrentMonthBillings"] != 0) else None, axis=1
    )
    merged = merged.drop(columns=["_key"], errors="ignore")
else:
    merged = pd.DataFrame(columns=["PropertyName","Current","Days30Plus","RunDate","CurrentMonthBillings","CurrentMonthBillingsNotCollected"])

out_path = TABLE_DIR / "ar_collections.xlsx"
merged.to_excel(str(out_path), index=False, engine="openpyxl")
print(f"\nar_collections -> {len(merged)} filas -> {out_path}")

In [ ]:
# ═══════════════════════════════════════════════════════
#  denies_cancells
#  Resident Activity - Denied / Cancelled files
# ═══════════════════════════════════════════════════════

files_dc = [
    p for p in SHEET_DIR.iterdir()
    if p.is_file()
    and re.search(r"resident_activity", p.name, re.I)
    and re.search(r"denied", p.name, re.I)
    and re.search(r"cancel", p.name, re.I)
]

print(f"Denied/Cancelled files: {len(files_dc)}")

RENAME_MAP = {
    "Bldg/Unit": "BldgUnit",
    "Building/Unit": "BldgUnit",
    "Cancelled/Denied Date": "CancelledDeniedDate",
    "Cancel/Denied Date": "CancelledDeniedDate",
    "Cancelled/Denied\nDate": "CancelledDeniedDate",
    "Cancel/Denied\nDate": "CancelledDeniedDate",
    "Cancelled/Denied Reason": "CancelledDeniedReason",
    "Cancel/Denied Reason": "CancelledDeniedReason",
    "Cancelled/Denied\nReason": "CancelledDeniedReason",
    "Cancel/Denied\nReason": "CancelledDeniedReason",
    "Leasing Consultant": "LeasingConsultant",
    "Deposits on Hand": "DepositsOnHand",
    "Deposits On Hand": "DepositsOnHand",
    "Deposits\nOn Hand": "DepositsOnHand",
    "Ledger Balance": "LedgerBalance",
    "Ledger\nBalance": "LedgerBalance",
}

OUT_COLS = ["Name","Status","BldgUnit","CancelledDeniedDate","CancelledDeniedReason",
            "LeasingConsultant","DepositsOnHand","LedgerBalance","Property","RunDate","bucket"]

dfs_dc = []
for fp in files_dc:
    try:
        raw = pd.read_excel(_long_path(fp), header=None, engine="openpyxl")

        # RunDate from row 1
        run_date = safe_to_date(raw.iloc[1, 0]) if len(raw) > 1 else None

        # Property from top 10
        top10 = raw.head(10).astype(str).values.flatten().tolist()
        hdr_line = next((x for x in top10 if " - " in x), None)
        prop = hdr_line.split(" - ", 1)[-1].strip() if hdr_line else fp.stem.split("_Resident")[0].replace("_", " ")

        if prop in EXCLUDE_PROPS:
            print(f"  SKIP {fp.name} (excluded)")
            continue

        # Find "Name" header row
        first_col = raw.iloc[:, 0].astype(str).tolist()
        try:
            hdr_idx = next(i for i, v in enumerate(first_col) if v.strip() == "Name")
        except StopIteration:
            print(f"  SKIP {fp.name}: no 'Name' header")
            continue

        df2 = raw.iloc[hdr_idx:].reset_index(drop=True)
        df2.columns = df2.iloc[0].astype(str)
        df2 = df2.iloc[1:].copy()

        # Filter blanks and totals
        name_col = df2.columns[0]
        df2 = df2[df2[name_col].notna()]
        df2 = df2[~df2[name_col].astype(str).str.strip().str.startswith("Total")]

        # Rename
        df2 = df2.rename(columns={k: v for k, v in RENAME_MAP.items() if k in df2.columns})

        # Ensure required columns exist
        for c in OUT_COLS[:-1]:
            if c not in df2.columns:
                df2[c] = None

        df2["CancelledDeniedDate"] = df2["CancelledDeniedDate"].apply(safe_to_date)
        df2["DepositsOnHand"]      = df2.get("DepositsOnHand", pd.Series(dtype=float)).apply(safe_float)
        df2["LedgerBalance"]       = df2.get("LedgerBalance",  pd.Series(dtype=float)).apply(safe_float)
        df2["Property"] = prop
        df2["RunDate"]  = str(run_date) if run_date else str(YESTERDAY)

        # Bucket
        cancel_kws = ["cancel", "changed their mind", "leased elsewhere"]
        def _bucket(reason):
            r = str(reason).lower()
            return "Cancelled" if any(k in r for k in cancel_kws) else "Denied"
        df2["bucket"] = df2.get("CancelledDeniedReason", pd.Series(dtype=str)).apply(_bucket)

        dfs_dc.append(df2[OUT_COLS])
        print(f"  OK  {fp.name} -> {prop}")
    except Exception as e:
        print(f"  ERR {fp.name}: {e}")

df_denies = pd.concat(dfs_dc, ignore_index=True) if dfs_dc else pd.DataFrame(columns=OUT_COLS)
out_path = TABLE_DIR / "denies_cancells.xlsx"
df_denies.to_excel(str(out_path), index=False, engine="openpyxl")
print(f"\ndenies_cancells -> {len(df_denies)} filas -> {out_path}")

In [ ]:
# ═══════════════════════════════════════════════════════
#  effective_rents
#  Rent Roll Detail Sheet2
# ═══════════════════════════════════════════════════════

EXCLUDE_ER = {
    "bbi_llc_-_single_familiy_rent_roll_detail_-_excel__sheet2.xlsx",
    "first_property_i_llc_-_single_family_rent_roll_detail_-_excel__sheet2.xlsx",
    "zoe_llc_rent_roll_detail_-_excel__sheet2.xlsx",
}

files_er = [
    p for p in SHEET_DIR.iterdir()
    if p.is_file()
    and "rent_roll_detail" in p.name.lower()
    and "sheet2" in p.name.lower()
    and p.name.lower() not in EXCLUDE_ER
]

print(f"Rent Roll Sheet2 files: {len(files_er)}")

def norm_hdr(x):
    if pd.isna(x): return ""
    s = str(x).replace("\n"," ").replace("\t"," ").strip()
    return re.sub(r"\s+", " ", s)

def find_fp_header(raw):
    for i in range(min(200, len(raw))):
        v = "" if pd.isna(raw.iloc[i, 0]) else str(raw.iloc[i, 0])
        if v.strip().lower() == "floorplan":
            return i
    return None

def parse_property(raw):
    for i in range(min(80, len(raw))):
        v = "" if pd.isna(raw.iloc[i, 0]) else str(raw.iloc[i, 0])
        if "heritage hill" in v.lower():
            if " - " in v: return v.split(" - ", 1)[-1].strip()
            if "-" in v:   return v.split("-", 1)[-1].strip()
            return v.strip()
    return None

def std_cols(pdf):
    cols = list(pdf.columns)
    normed = [norm_hdr(c).lower() for c in cols]
    def pick(pred):
        for orig, n in zip(cols, normed):
            if pred(n): return orig
        return None
    rm = {}
    fp  = pick(lambda n: n == "floorplan" or "floorplan" in n)
    if fp: rm[fp] = "Floorplan"
    uo  = pick(lambda n: "unit" in n and "occup" in n)
    if uo: rm[uo] = "Units_Occupied"
    al  = pick(lambda n: "average" in n and "leas" in n)
    if al: rm[al] = "Average_Leased"
    nu  = pick(lambda n: ("unit" in n and "#" in n) or n in ("# units","units"))
    if nu: rm[nu] = "Num_Units"
    oc  = pick(lambda n: "occup" in n and "%" in n)
    if oc: rm[oc] = "Occupancy_Pct"
    return pdf.rename(columns=rm)

dfs_er = []
for fp in files_er:
    try:
        raw = pd.read_excel(_long_path(fp), header=None, engine="openpyxl")
        hdr_idx = find_fp_header(raw)
        if hdr_idx is None:
            print(f"  SKIP {fp.name}: no Floorplan header")
            continue

        prop = parse_property(raw)

        headers = [norm_hdr(x) for x in raw.iloc[hdr_idx].tolist()]
        data = raw.iloc[hdr_idx + 1:].copy()
        data.columns = headers
        data = std_cols(data)

        for c in ["Floorplan","Units_Occupied","Average_Leased","Num_Units","Occupancy_Pct"]:
            if c not in data.columns: data[c] = None

        data["Property"] = prop

        # Clean Floorplan
        data["Floorplan"] = data["Floorplan"].astype(str).str.strip()
        data = data[data["Floorplan"].notna() & (data["Floorplan"] != "") & (data["Floorplan"] != "nan")]
        data = data[~data["Floorplan"].str.lower().str.startswith("total")]

        # Bedroom
        def get_bedroom(fp_val):
            s = str(fp_val).strip().lower()
            if re.match(r"^(studio|s)\b", s): return "0"
            m = re.search(r"(\d+)", re.sub(r"^[^0-9]+", "", str(fp_val)))
            return m.group(1) if m else ""
        data["Bedroom"] = data["Floorplan"].apply(get_bedroom)
        data = data[data["Bedroom"].notna() & (data["Bedroom"] != "")]
        data = data[~data["Bedroom"].str.upper().isin(["T","R","P","H","F","A"])]

        # Unit_Type
        data["Unit_Type"] = data["Bedroom"].apply(
            lambda b: "Studio Rent" if b == "0" else f"{b} Bed Rent"
        )

        # Parse numbers
        data["Units_Occupied"] = pd.to_numeric(
            data["Units_Occupied"].astype(str).str.replace(",",""), errors="coerce"
        ).fillna(0).astype(int)
        data["Average_Leased"] = pd.to_numeric(
            data["Average_Leased"].astype(str).str.replace(",","").str.replace("%",""), errors="coerce"
        )

        data = data[data["Average_Leased"].notna() & (data["Average_Leased"] != 0)]
        data = data[data["Units_Occupied"] > 0]
        data = data[data["Property"].notna() & (data["Property"].astype(str).str.strip() != "")]

        data["SumProduct"] = data["Average_Leased"] * data["Units_Occupied"]
        dfs_er.append(data)
        print(f"  OK  {fp.name} -> {prop}")
    except Exception as e:
        print(f"  ERR {fp.name}: {e}")

if dfs_er:
    combined = pd.concat(dfs_er, ignore_index=True)
    grp = combined.groupby(["Property","Unit_Type"]).agg(
        SumProduct    =("SumProduct","sum"),
        Units_Occupied=("Units_Occupied","sum")
    ).reset_index()
    grp["Effective_Rent"] = (grp["SumProduct"] / grp["Units_Occupied"]).round(0)
    grp["RunDate"] = str(YESTERDAY)
    result_er = grp[["RunDate","Property","Unit_Type","Units_Occupied","Effective_Rent"]]
else:
    result_er = pd.DataFrame(columns=["RunDate","Property","Unit_Type","Units_Occupied","Effective_Rent"])

out_path = TABLE_DIR / "effective_rents.xlsx"
result_er.to_excel(str(out_path), index=False, engine="openpyxl")
print(f"\neffective_rents -> {len(result_er)} filas -> {out_path}")

In [ ]:
# ═══════════════════════════════════════════════════════
#  move_ins
#  Resident Activity - MOVE-INS files
# ═══════════════════════════════════════════════════════

EXCLUDE_MI = {
    "bbi_llc_-_single_familiy_resident_activity_-_excel__move-ins.xlsx",
    "bbi_llc_-_single_family_resident_activity_-_excel_move-ins.xlsx",
    "first_property_i_llc_-_single_family_resident_activity_-_excel_move-ins.xlsx",
    "zoe_llc_resident_activity_-_excel_move-ins.xlsx",
}

files_mi = [
    p for p in SHEET_DIR.iterdir()
    if p.is_file()
    and "resident_activity" in p.name.lower()
    and "move-ins" in p.name.lower()
    and p.name.lower() not in EXCLUDE_MI
]

print(f"MOVE-INS files: {len(files_mi)}")

FINAL_COLS = {
    "Name": "Name",
    "Status": "Status",
    "Bldg/Unit": "Bldg_Unit",
    "Lease Rent": "Lease_Rent",
    "Other Recurring Charges": "Other_Recurring_Charges",
    "Other Recurring Credits": "Other_Recurring_Credits",
    "Ad Source": "Ad_Source",
    "Move - In Date": "Move_In_Date",
    "Lease End Date": "Lease_End_Date",
    "Leasing Consultant": "Leasing_Consultant",
    "MOVE-INS": "MOVE_INS",
    "Property": "Property",
    "RunDate": "RunDate",
}
NUM_COLS  = ["Lease Rent","Other Recurring Charges","Other Recurring Credits"]
DATE_COLS = ["Move - In Date","Lease End Date"]

dfs_mi = []
for fp in files_mi:
    try:
        raw = pd.read_excel(_long_path(fp), header=None, engine="openpyxl")

        # RunDate from row 1
        run_date = safe_to_date(raw.iloc[1, 0]) if len(raw) > 1 else None

        # Property from top 10
        top10 = raw.head(10).astype(str).values.flatten().tolist()
        hdr_line = next((x for x in top10 if " - " in x), None)
        prop = hdr_line.split(" - ", 1)[1].strip() if hdr_line else fp.stem.split("_Resident")[0].replace("_"," ")

        if prop in EXCLUDE_PROPS:
            print(f"  SKIP {fp.name} (excluded)")
            continue

        # Find "Name" header row
        first_col = raw.iloc[:, 0].astype(str).tolist()
        try:
            hdr_idx = next(i for i, v in enumerate(first_col) if v.strip() == "Name")
        except StopIteration:
            print(f"  SKIP {fp.name}: no 'Name' header")
            continue

        df2 = raw.iloc[hdr_idx:].reset_index(drop=True)
        df2.columns = df2.iloc[0].astype(str).str.replace("\n"," ").str.replace("\r"," ").str.strip()
        df2 = df2.iloc[1:].copy()

        # Ensure columns
        for c in FINAL_COLS:
            if c not in df2.columns: df2[c] = None

        df2 = df2[list(FINAL_COLS.keys())].copy()

        # Filter
        if "Name" in df2.columns:
            df2 = df2[df2["Name"].notna()]
            df2 = df2[~df2["Name"].astype(str).str.startswith("Total", na=False)]

        # Parse numbers/dates
        for c in NUM_COLS:
            if c in df2.columns:
                df2[c] = pd.to_numeric(df2[c], errors="coerce")
        for c in DATE_COLS:
            if c in df2.columns:
                df2[c] = pd.to_datetime(df2[c], errors="coerce").dt.date.apply(
                    lambda x: x.strftime("%Y-%m-%d") if x else None
                )

        df2["Property"] = prop
        df2["RunDate"]  = str(run_date) if run_date else str(YESTERDAY)

        # Rename
        df2 = df2.rename(columns=FINAL_COLS)

        dfs_mi.append(df2)
        print(f"  OK  {fp.name} -> {prop}")
    except Exception as e:
        print(f"  ERR {fp.name}: {e}")

df_mi = pd.concat(dfs_mi, ignore_index=True) if dfs_mi else pd.DataFrame(columns=list(FINAL_COLS.values()))
out_path = TABLE_DIR / "move_ins.xlsx"
df_mi.to_excel(str(out_path), index=False, engine="openpyxl")
print(f"\nmove_ins -> {len(df_mi)} filas -> {out_path}")

In [ ]:
# ═══════════════════════════════════════════════════════
#  deliquent_prepaid
#  Delinquent Sheet2 (main) + Sheet1 (Rent-S8 join)
# ═══════════════════════════════════════════════════════

files_s2 = [
    p for p in SHEET_DIR.iterdir()
    if p.is_file()
    and "delinquent_and_prepaid" in p.name.lower()
    and "sheet2" in p.name.lower()
]
files_s1 = [
    p for p in SHEET_DIR.iterdir()
    if p.is_file()
    and "delinquent_and_prepaid" in p.name.lower()
    and "sheet1" in p.name.lower()
]
print(f"Sheet2 files: {len(files_s2)}")
print(f"Sheet1 files: {len(files_s1)}")

def parse_banner_prop(raw, fname):
    for i in range(min(3, len(raw))):
        v = raw.iloc[i, 0]
        if v is None or pd.isna(v): continue
        banner = str(v).replace("\r","")
        lines = [l.strip() for l in banner.split("\n") if l.strip()]
        if lines:
            parts = lines[0].split(" - ")
            if len(parts) >= 2: return parts[1].strip()
        break
    return fname.split("_delinquent_and_prepaid")[0].replace("_"," ")

def find_hdr_row_tokens(raw, tokens, scan=100):
    for i in range(min(scan, len(raw))):
        row_vals = [str(v).strip() if not pd.isna(v) else "" for v in raw.iloc[i]]
        if all(tok in row_vals for tok in tokens):
            return i
    return -1

# ─── Process Sheet2 ────────────────────────────────────────────────────────
def process_sheet2(fp):
    raw = pd.read_excel(_long_path(fp), header=None, engine="openpyxl")
    prop = parse_banner_prop(raw, fp.stem.lower())

    hdr_idx = None
    for i in range(len(raw)):
        v = raw.iloc[i, 0]
        if not pd.isna(v) and str(v).strip() == "Resh ID":
            hdr_idx = i; break
    if hdr_idx is None: return None

    df = raw.iloc[hdr_idx:].copy()
    df.columns = df.iloc[0].apply(lambda x: "" if pd.isna(x) else str(x).strip())
    df = df.iloc[1:].reset_index(drop=True)

    NEED = ["Resh ID","Lease ID","Bldg/Unit","Name","Phone Number","Email","Status","Move-In/Out",
            "Total Prepaid","Total Delinquent","Net Balance","Current","30 Days","60 Days","90+ Days",
            "Prorate Credit","Deposits Held","Outstanding Deposit","#Late","#NSF",
            "Delinquency Comment","Comment Date"]
    for n in NEED:
        if n not in df.columns: df[n] = None

    df = df[df["Resh ID"].notna() & df["Lease ID"].notna() & df["Bldg/Unit"].notna()]

    num_cols = ["Total Prepaid","Total Delinquent","Net Balance","Current",
                "30 Days","60 Days","90+ Days","Prorate Credit","Deposits Held","Outstanding Deposit",
                "Resh ID","Lease ID","#Late","#NSF"]
    for c in num_cols:
        df[c] = pd.to_numeric(df[c], errors="coerce")

    df["Comment Date"] = pd.to_datetime(df["Comment Date"], errors="coerce").dt.date.apply(
        lambda x: x.strftime("%Y-%m-%d") if pd.notna(x) and hasattr(x,"strftime") else None
    )

    df["Property"] = prop
    df["JoinName"] = df["Name"].apply(join_key)
    return df

# ─── Process Sheet1 (Rent-S8 map) ─────────────────────────────────────────
def process_sheet1_s8(fp):
    raw = pd.read_excel(_long_path(fp), header=None, engine="openpyxl")
    prop = parse_banner_prop(raw, fp.stem.lower())

    hdr = find_hdr_row_tokens(raw, ["Name","Code Description"])
    if hdr == -1: return None

    t = raw.iloc[hdr:].copy()
    t.columns = t.iloc[0].apply(lambda x: "" if pd.isna(x) else str(x).strip())
    t = t.iloc[1:].reset_index(drop=True)

    if "Name" not in t.columns or "Code Description" not in t.columns or "Total Delinquent" not in t.columns:
        return None

    t["Property"] = prop
    t["JoinName"] = t["Name"].apply(join_key)
    t = t[t["Code Description"].astype(str).str.contains("Rent - S8", case=False, na=False)]
    t["S8Delinquent"] = pd.to_numeric(t["Total Delinquent"], errors="coerce")

    return t.groupby(["Property","JoinName"])["S8Delinquent"].sum(min_count=1).reset_index()             .rename(columns={"S8Delinquent":"total_delinquent_rent_s8"})

# ─── Build Sheet2 dataframe ────────────────────────────────────────────────
pdfs2 = []
for fp in files_s2:
    try:
        r = process_sheet2(fp)
        if r is not None:
            pdfs2.append(r)
            print(f"  S2 OK  {fp.name}")
    except Exception as e:
        print(f"  S2 ERR {fp.name}: {e}")

if not pdfs2:
    raise RuntimeError("No Sheet2 files could be parsed for deliquent_prepaid")

pdf = pd.concat(pdfs2, ignore_index=True)

# ─── Build Sheet1 s8 map ──────────────────────────────────────────────────
maps = []
for fp in files_s1:
    try:
        r = process_sheet1_s8(fp)
        if r is not None:
            maps.append(r)
            print(f"  S1 OK  {fp.name}")
    except Exception as e:
        print(f"  S1 ERR {fp.name}: {e}")

if maps:
    s8_map = pd.concat(maps, ignore_index=True)
    pdf = pdf.merge(s8_map, how="left", on=["Property","JoinName"])
else:
    pdf["total_delinquent_rent_s8"] = None

# ─── Rename + rundate ─────────────────────────────────────────────────────
pdf.rename(columns={
    "Resh ID":"resh_id","Lease ID":"lease_id","Bldg/Unit":"bldg_unit",
    "Name":"name","Phone Number":"phone_number","Email":"email",
    "Status":"status","Move-In/Out":"move_in_out",
    "Total Prepaid":"total_prepaid","Total Delinquent":"total_delinquent",
    "Net Balance":"net_balance","Current":"current",
    "30 Days":"days_30","60 Days":"days_60","90+ Days":"days_90_plus",
    "Prorate Credit":"prorate_credit","Deposits Held":"deposits_held",
    "Outstanding Deposit":"outstanding_deposit",
    "#Late":"late_count","#NSF":"nsf_count",
    "Delinquency Comment":"delinquency_comment","Comment Date":"comment_date",
    "Property":"property"
}, inplace=True)

pdf["rundate"] = str(YESTERDAY)

def days_since(date_str):
    try:
        d = dt.date.fromisoformat(str(date_str))
        diff = (TODAY - d).days
        return max(diff, 0)
    except Exception:
        return None

pdf["days_since_comment_date"] = pdf["comment_date"].apply(days_since)

# Drop helper column
pdf = pdf.drop(columns=["JoinName"], errors="ignore")

out_path = TABLE_DIR / "deliquent_prepaid.xlsx"
pdf.to_excel(str(out_path), index=False, engine="openpyxl")
print(f"\ndeliquent_prepaid -> {len(pdf)} filas -> {out_path}")

In [ ]:
# ═══════════════════════════════════════════════════════
#  leasing_desk
#  Applicant_Detail files
# ═══════════════════════════════════════════════════════

files_ld = [
    p for p in SHEET_DIR.iterdir()
    if p.is_file()
    and re.search(r"applicant_detail__applicant_detail", p.name, re.I)
]

print(f"Applicant_Detail files: {len(files_ld)}")

def clean_cols(cols):
    out, seen = [], {}
    for i, c in enumerate(cols, start=1):
        if c is None or (isinstance(c, float) and math.isnan(c)):
            c = f"Column{i}"
        c = str(c).strip()
        if c == "" or c.lower() == "nan" or c.lower().startswith("unnamed"):
            c = f"Column{i}"
        if c in seen:
            seen[c] += 1
            c = f"{c}_{seen[c]}"
        else:
            seen[c] = 1
        out.append(c)
    return out

def delta_safe(name):
    s = str(name).strip()
    s = re.sub(r"[ ,;{}()\n\t=]", "_", s)
    s = re.sub(r"[^0-9A-Za-z_]", "", s)
    s = re.sub(r"_+", "_", s).strip("_")
    if not s or s[0].isdigit():
        s = f"col_{s}" if s else "col"
    return s

DATE_COLS_LD = ["Movein Date","Decision Date","App Result Date","App Submission Date"]

dfs_ld = []
for fp in files_ld:
    try:
        raw = pd.read_excel(_long_path(fp), sheet_name=0, header=None, engine="openpyxl")

        # Find first non-null row (actual header)
        raw = raw[raw.iloc[:, 0].notna()]
        raw = raw[raw.iloc[:, 0].astype(str) != "Unnamed: 0"]

        if raw.empty:
            print(f"  SKIP {fp.name}: empty after filter")
            continue

        header = list(raw.iloc[0].values)
        data = raw.iloc[1:].reset_index(drop=True)
        data.columns = clean_cols(header)

        data = data.dropna(axis=1, how="all")

        # Format date columns
        for c in DATE_COLS_LD:
            if c in data.columns:
                dt_series = pd.to_datetime(data[c], errors="coerce")
                data[c] = dt_series.dt.strftime("%Y-%m-%d")

        # Convert complex types to string
        for c in data.columns:
            if data[c].apply(lambda x: isinstance(x, (dict, list, tuple))).any():
                data[c] = data[c].astype(str)

        # Rename columns to delta-safe names
        orig_cols = list(data.columns)
        safe_names, seen = [], {}
        for c in orig_cols:
            s = delta_safe(c)
            if s in seen:
                seen[s] += 1
                s = f"{s}_{seen[s]}"
            else:
                seen[s] = 1
            safe_names.append(s)

        data.columns = safe_names

        data["SourceFile"] = fp.name
        data["RunDate"]    = str(TODAY)

        dfs_ld.append(data)
        print(f"  OK  {fp.name}")
    except Exception as e:
        print(f"  ERR {fp.name}: {e}")

if dfs_ld:
    df_ld = pd.concat(dfs_ld, ignore_index=True)
    # Drop LDSScore rows that are non-numeric
    if "LDSScore" in df_ld.columns:
        def is_bad_lds(x):
            s = str(x).strip().lower()
            if s in ("no tradelines","no records"): return True
            try: float(s); return False
            except: return True
        df_ld = df_ld[~df_ld["LDSScore"].apply(is_bad_lds)]
else:
    df_ld = pd.DataFrame()

out_path = TABLE_DIR / "leasing_desk.xlsx"
df_ld.to_excel(str(out_path), index=False, engine="openpyxl")
print(f"\nleasing_desk -> {len(df_ld)} filas -> {out_path}")

In [ ]:
import os
print("\n" + "="*55)
print("  PIPELINE COMPLETO")
print("="*55)
outputs = [
    ("ar_collections",    "ar_collections.xlsx"),
    ("denies_cancells",   "denies_cancells.xlsx"),
    ("effective_rents",   "effective_rents.xlsx"),
    ("move_ins",          "move_ins.xlsx"),
    ("deliquent_prepaid", "deliquent_prepaid.xlsx"),
    ("leasing_desk",      "leasing_desk.xlsx"),
]
for label, fname in outputs:
    p = TABLE_DIR / fname
    if p.exists():
        size_kb = p.stat().st_size / 1024
        df_check = pd.read_excel(str(p), engine="openpyxl")
        print(f"  {label:<22} {len(df_check):>5} filas  ({size_kb:.1f} KB)")
    else:
        print(f"  {label:<22} NO GENERADO")
print("="*55)
print(f"  Tablas en: {TABLE_DIR}")